# Local NeMo Gym Rollout

This notebook runs the local NeMo Gym environment using the Relay-derived data, the `relay_handoff` resources server, and the scripted local Responses API model.

In [ ]:
from pathlib import Path
import json

ROOT = Path.cwd()
ROLLUP = ROOT / "outputs" / "nemo_gym" / "rollouts.jsonl"
METRICS = ROOT / "outputs" / "nemo_gym" / "rollouts_aggregate_metrics.json"

Validate the merged NeMo Gym config. This command uses the local resources server and local scripted model server.

In [ ]:
! PATH="$PWD/.venv-nemo-gym/bin:$PATH" .venv-nemo-gym/bin/gym env validate --config resources_servers/relay_handoff/configs/relay_handoff.yaml --config responses_api_models/scripted_handoff_model/configs/scripted_handoff_model.yaml --search-dir .

Collate the NeMo Gym train and validation JSONL.

In [ ]:
! PATH="$PWD/.venv-nemo-gym/bin:$PATH" .venv-nemo-gym/bin/gym dataset collate --config resources_servers/relay_handoff/configs/relay_handoff.yaml --resources-server relay_handoff --mode train_preparation --output-dir outputs/nemo_gym/collated

Run a local validation rollout. The output should show `reward=1.0` and `handoff_order_ok=true` for the scripted happy path.

In [ ]:
! PATH="$PWD/.venv-nemo-gym/bin:$PATH" .venv-nemo-gym/bin/gym eval run --config resources_servers/relay_handoff/configs/relay_handoff.yaml --config responses_api_models/scripted_handoff_model/configs/scripted_handoff_model.yaml --search-dir . --agent relay_handoff_simple_agent --input resources_servers/relay_handoff/data/validation.jsonl --output outputs/nemo_gym/rollouts.jsonl --limit 1 --num-repeats 1 --split validation +upload_rollouts_to_wandb=false

In [ ]:
with ROLLUP.open() as f:
    rollout = json.loads(next(f))

{
    "reward": rollout["reward"],
    "actual_handoff_tools": rollout["actual_handoff_tools"],
    "missing_handoff_tools": rollout["missing_handoff_tools"],
    "handoff_order_ok": rollout["handoff_order_ok"],
}

In [ ]:
json.loads(METRICS.read_text())[0]["key_metrics"]